In [1]:
from pydantic import BaseModel, Field
from typing import List
from openai import OpenAI
import pandas as pd
from dotenv import load_dotenv
import sys
import os
sys.path.append(os.path.abspath('..')) # Adds the parent directory to the path

import time
import csv
from langchain_chroma import Chroma
from embedder import Embedder
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
embeddings = Embedder(path="../models/Xenova/all-MiniLM-L6-v2")

PERSIST_DIR = os.getenv("CHROMA_DB_DIR")

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=os.getenv("MODEL_BASE_URL")
)

class VehicleIssue(BaseModel):
    issue_id: str = Field(description="Unique identifier, e.g. 'p0420-001'")
    issue_name: str = Field(description="Name of the issue, e.g. 'Catalyst System Efficiency Below Threshold'")
    obd_code: str = Field(description="Real OBD-II code, e.g. 'P0420', 'P0300'. Use 'N/A' if not code-based")
    system: str = Field(description="Engine, Transmission, Brakes, Electrical, Suspension, Cooling, etc.")
    component: str = Field(description="Specific part, e.g. 'Catalytic Converter', 'Brake Pads', 'Alternator'")
    severity: str = Field(description="Low, Medium, High, Critical")
    symptoms: str = Field(description="Comma-separated list, e.g. 'Check engine light, rough idle, reduced power'")
    likely_causes: str = Field(description="Comma-separated list, e.g. 'Faulty O2 sensor, worn spark plugs'")
    diagnostic_steps: str = Field(description="Step-by-step instructions to confirm the diagnosis")
    diy_or_mechanic: str = Field(description="DIY, Mechanic Recommended, Mechanic Required")

class VehicleIssueDataset(BaseModel):
    issues: List[VehicleIssue]

In [2]:
def normalize_text_result(doc):
    return {
        "issue_id": doc.get("issue_id", ""),
        "content": (
            f"Issue: {doc.get('issue_name','')}\n"
            f"Symptoms: {doc.get('symptoms','')}\n"
            f"Likely Causes: {doc.get('likely_causes','')}\n"
            f"Diagnostic Steps: {doc.get('diagnostic_steps','')}"
        ),
        "source": "text_search"
    }
def normalize_vector_result(doc):
    return {
        "issue_id": doc.metadata.get("issue_id", ""),
        "content": doc.page_content,
        "source": "vector_search"
    }

In [3]:
def rrf(search_results, k=1, num_results=10):
    scores = {}
    doc_map = {}
    for results in search_results:
        for rank, doc in enumerate(results):
            key = doc["issue_id"]
            if key not in scores:
                scores[key] = 0
                doc_map[key] = doc
            scores[key] += 1 / (k + rank + 1)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_map[key] for key, _ in ranked[:num_results]]

In [4]:
def hybrid_search(query, num_results=10):
    text_results = [normalize_text_result(r) for r in search(query)]
    vector_results = [normalize_vector_result(d) for d in retrieve_documents(query)]
    return rrf([text_results, vector_results], num_results=num_results)

In [6]:
all_issues = []
batch_size = 10
num_batches = 5

def generate_batch(batch_prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = openai_client.chat.completions.parse(
                model=os.getenv("AI_MODEL"),
                messages=[{"role": "user", "content": batch_prompt}],
                response_format=VehicleIssueDataset,
                reasoning_effort="low",
            )
            return response.choices[0].message.parsed.issues
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(1)
    print("  All retries failed, skipping batch.")
    return []

all_issues = []
batch_size = 10
num_batches = 5

for i in range(num_batches):
    batch_prompt = f"""
Generate exactly {batch_size} diverse, realistic vehicle issues (batch {i+1} of {num_batches}).
Cover different systems (engine, transmission, brakes, electrical, suspension, cooling).
Use real OBD-II codes where applicable (e.g. P0171, P0300, P0420, P0455), 'N/A' for non-code issues.
Vary severity levels and mix DIY-fixable issues with ones that require a mechanic.
Output valid JSON only — field names must exactly match the schema, with no markdown formatting (no asterisks, no bold) in keys or values.
Avoid repeating issues already generated: {[iss.issue_name for iss in all_issues]}
""".strip()

    batch_issues = generate_batch(batch_prompt)
    print(f"Batch {i+1}: requested {batch_size}, got {len(batch_issues)}")
    all_issues.extend(batch_issues)

df = pd.DataFrame([issue.model_dump() for issue in all_issues])
df.to_csv("data/data.csv", index=False)
print(f"Generated {len(df)} vehicle issues")

Batch 1: requested 10, got 10
Batch 2: requested 10, got 10
Batch 3: requested 10, got 10
Batch 4: requested 10, got 10
Batch 5: requested 10, got 10
Generated 50 vehicle issues


In [7]:
from sqlitesearch import TextSearchIndex
index = TextSearchIndex(
    text_fields=[
        "issue_name",
        "system",
        "component",
        "symptoms",
        "likely_causes",
        "diagnostic_steps",
    ],
    keyword_fields=["issue_id"],
    db_path=os.getenv("SQLITESEARCHDB")
)

In [8]:
import pandas as pd
df = pd.read_csv("data/data.csv")
documents = df.to_dict(orient="records")

for doc in documents:
    index.add(doc)
    print(f"""Added: {doc["issue_name"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to issues.db")

Added: System Too Lean (Bank 1)...
Added: Random/Multiple Cylinder Misfire Detected...
Added: Catalyst System Efficiency Below Threshold (Bank 1)...
Added: Evaporative Emission System Leak Detected (Large Leak)...
Added: Worn Brake Pads...
Added: Delayed Gear Engagement...
Added: Battery Terminal Corrosion...
Added: Worn Front Shock Absorbers...
Added: Coolant Leak at Radiator Hose...
Added: Faulty Interior Lighting Relay...
Added: System Too Lean (Bank 1)...
Added: Random/Multiple Cylinder Misfire Detected...
Added: Catalyst System Efficiency Below Threshold (Bank 1)...
Added: Evaporative Emission System Leak Detected (Large Leak)...
Added: Worn Brake Pads...
Added: Delayed Gear Engagement...
Added: Battery Terminal Corrosion...
Added: Worn Front Shock Absorbers...
Added: Coolant Leak at Radiator Hose...
Added: Faulty Interior Lighting Relay...
Added: System Too Lean (Bank 2)...
Added: Cylinder 1 Misfire Detected...
Added: Coolant Thermostat Malfunction...
Added: Vehicle Speed Sensor 

In [9]:
index.count()

50

In [10]:


def create_obd_vectorstore(csv_path: str):
    """Create a vector store from the OBD diagnostic CSV."""
    documents = []
      # If the DB already exists on disk, just load it — don't re-embed everything again
    if os.path.exists(PERSIST_DIR):
        return Chroma(
            collection_name=os.getenv("CHROMA_COLLECTION"),
            embedding_function=embeddings,
            persist_directory=PERSIST_DIR,
        )

    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Build page_content from the fields useful for semantic search
            page_content = f"""
            Issue: {row['issue_name']} ({row['obd_code']})
            System: {row['system']} | Component: {row['component']} | Severity: {row['severity']}
            Symptoms: {row['symptoms']}
            Likely Causes: {row['likely_causes']}
            Diagnostic Steps: {row['diagnostic_steps']}
            Recommendation: {row['diy_or_mechanic']}
            """.strip()

            metadata = {
                "issue_id": row["issue_id"],
                "obd_code": row["obd_code"],
                "system": row["system"],
                "component": row["component"],
                "severity": row["severity"],
                "diy_or_mechanic": row["diy_or_mechanic"],
                "source": csv_path,
            }

            documents.append(Document(page_content=page_content, metadata=metadata))

    vectorstore = Chroma.from_documents(
        documents=documents, embedding=embeddings, collection_name=os.getenv("CHROMA_COLLECTION"),
        persist_directory=PERSIST_DIR,

    )

    return vectorstore
create_obd_vectorstore("data/data.csv")


In [11]:
def search(query):
    boost = {'issue_name': 2.97871337674074, 
    'obd_code': 0.8807282883353911, 
    'system': 2.2496974218765224, 
    'component': 2.859062539433551, 
    'severity': 1.1868442875376763, 
    'symptoms': 1.712188340680863, 
    'likely_causes': 2.0352532451104133, 
    'diagnostic_steps': 2.114980686414594, 
    'diy_or_mechanic': 1.0735542596387404}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=5
    )

    return results

In [12]:
prompt_template = """
You're a vehicle diagnostic assistant. Answer the QUESTION based on the CONTEXT from our vehicle issues database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

def build_prompt(query, search_results):
    context = "\n---\n".join(doc["content"] for doc in search_results)
    return prompt_template.format(question=query, context=context).strip()

In [13]:
from openai import OpenAI

def llm(prompt, model=os.getenv("AI_MODEL")):
    response = openai_client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}]
    )

    return response.output_text

In [14]:
def load_obd_vectorstore():
    """Load the existing persisted OBD vector store."""
    return Chroma(
        collection_name=os.getenv("CHROMA_COLLECTION"),
        embedding_function=embeddings,
        persist_directory=PERSIST_DIR,
    )


def retrieve_documents(query: str, vectorstore=None) -> list:
    """
    Retrieve documents based on the query.
    """

    if not vectorstore:
        vectorstore = load_obd_vectorstore()

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    documents = retriever.invoke(query)
    
    return documents

# vectorstore = load_obd_vectorstore()
# docs = retrieve_documents("engine shaking and losing power", vectorstore=vectorstore)

In [15]:
# In a new cell:
collection = load_obd_vectorstore()._collection
print(f"Total entries in 'obd_diagnostics': {collection.count()}")

Total entries in 'obd_diagnostics': 50


In [16]:
def rag(query, model=os.getenv("AI_MODEL")):
    combined_results = hybrid_search(query)
    prompt = build_prompt(query, combined_results)
    answer = llm(prompt, model=model)
    return answer

In [17]:
question = "Why does my car make a clunking sound when I go over bumps?"
answer = rag(question)
print(answer)

A clunking noise that appears when you drive over bumps is most often a sign that something in the suspension that should be holding the chassis steady is worn out.  

**According to the data you provided, the two most common reasons are:**

| Possible Issue | Why it makes a clunk | Typical accompanying signs |
|----------------|----------------------|----------------------------|
| **Worn front shock absorbers** | The shock absorber’s internal seals and pistons lose their ability to control the spring’s motion. When a wheel hits a bump the spring can move freely, and the metal‑to‑metal contact in the shock or its mounting points produces a distinct “clunk.” | Excessive body roll, uneven front‑tire wear, sagging ride height, and oil leaks on the shock body. |
| **Rear upper control‑arm bush wear** | The rubber bushings that isolate the control arm from the frame degrade (crack, harden, or become loose). Over a bump the control arm can shift slightly before the worn bushing settles, cre

In [18]:
prompt1_template = """
You are a vehicle diagnostic expert generating evaluation questions.
For the issue below, generate 2 questions that a user might ask.
Return the result as a JSON array with objects containing 'issue_id' and 'question' fields.
Issue: {issue_name}
OBD Code: {obd_code}
System: {system}
Component: {component}
Symptoms: {symptoms}
Likely causes: {likely_causes}
Diagnostic steps: {diagnostic_steps}
""".strip()
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=os.getenv("MODEL_BASE_URL")
)

from tqdm.auto import tqdm
import json

df = pd.read_csv("data/data.csv")

results = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    prompt = prompt1_template.format(**row.to_dict())
    response = openai_client.chat.completions.create(
        model=os.getenv("AI_MODEL"),
        messages=[{"role": "user", "content": prompt}]
    )
    print(repr(response.choices[0].message.content))
    content = response.choices[0].message.content.strip()

    # strip markdown code fences if the model added them
    if content.startswith("```"):
        content = content.strip("`")
        if content.startswith("json"):
            content = content[4:].strip()

    try:
        questions = json.loads(content)
    except json.JSONDecodeError:
        print(f"Failed to parse issue_id={row['issue_id']}: {content[:200]}")
        continue

    for q in questions:
        q["issue_id"] = row["issue_id"]
        results.append(q)

df_questions = pd.DataFrame(results)
df_questions.to_csv("data/ground-truth-retrieval.csv", index=False)

  0%|          | 0/50 [00:00<?, ?it/s]

'[\n  {\n    "issue_id": "P0171",\n    "question": "What are the most common reasons for a P0171 code and how can I tell if it\'s a vacuum leak or a dirty MAF sensor?"\n  },\n  {\n    "issue_id": "P0171",\n    "question": "How do I test fuel pressure and fuel trim values to determine if a weak fuel pump is causing the system‑too‑lean condition?"\n  }\n]'
'[\n  {\n    "issue_id": "P0300",\n    "question": "What are the most common reasons for a random/multiple cylinder misfire and how can I tell if my spark plugs or ignition coils are the culprit?"\n  },\n  {\n    "issue_id": "P0300",\n    "question": "How do I check the fuel pressure on my vehicle and what pressure reading indicates a problem that could cause a P0300 code?"\n  }\n]'
'[\n  {\n    "issue_id": "P0420",\n    "question": "What are the most common causes of a P0420 code and how can I determine which one is affecting my vehicle?"\n  },\n  {\n    "issue_id": "P0420",\n    "question": "Can a faulty O2 sensor trigger a P0420, an

In [19]:
import pandas as pd
from tqdm.auto import tqdm

df_question = pd.read_csv("data/ground-truth-retrieval.csv")
ground_truth = df_question.to_dict(orient="records")

In [20]:
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line:
            cnt += 1
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank]:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        doc_id = q["issue_id"]
        results = search_function(q)
        relevance = [d["issue_id"] == doc_id for d in results]
        relevance_total.append(relevance)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [21]:
evaluate(ground_truth, lambda q: search(q["question"]))


  0%|          | 0/100 [00:00<?, ?it/s]

{'hit_rate': 0.97, 'mrr': 0.7958333333333333}

In [22]:
df_validation = df_question[:32]
df_test = df_question[32:]

gt_val = df_validation.to_dict(orient="records")
gt_test = df_test.to_dict(orient="records")

In [23]:
print(len(df_test))

68


In [24]:
def sqllite_search(query, boost=None):
    if boost is None:
        boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=5
    )
    return results

In [25]:
import random

def simple_optimize(param_ranges, objective_function, n_iterations=10):
    best_params = None
    best_score = float("-inf")

    for _ in range(n_iterations):
        current_params = {}
        for field, (low, high) in param_ranges.items():
            current_params[field] = random.uniform(low, high)

        current_score = objective_function(current_params)

        if current_score > best_score:
            best_score = current_score
            best_params = current_params

    return best_params

param_ranges = {
    "issue_name": (0.0, 3.0),
    "obd_code": (0.0, 3.0),
    "system": (0.0, 3.0),
    "component": (0.0, 3.0),
    "severity": (0.0, 3.0),
    "symptoms": (0.0, 3.0),
    "likely_causes": (0.0, 3.0),
    "diagnostic_steps": (0.0, 3.0),
    "diy_or_mechanic": (0.0, 3.0),
}
def objective(boost_params):
    def search_function(q):
        return sqllite_search(q["question"], boost=boost_params)

    results = evaluate(gt_val, search_function)
    return results["hit_rate"]

best_params = simple_optimize(param_ranges, objective, n_iterations=20)
print("Best boost parameters:", best_params)

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

...

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/32 [00:00<?, ?it/s]

Best boost parameters: {'issue_name': 2.9774666070614675, 'obd_code': 0.848388414206867, 'system': 2.5958420896162804, 'component': 0.6540060038455323, 'severity': 1.984634468965897, 'symptoms': 0.44788373658249625, 'likely_causes': 0.44860245116828357, 'diagnostic_steps': 1.8985348253693604, 'diy_or_mechanic': 1.4924207885021397}


In [26]:
def search(query):
    boost = {'issue_name': 2.9774666070614675, 'obd_code': 0.848388414206867, 'system': 2.5958420896162804, 'component': 0.6540060038455323, 'severity': 1.984634468965897, 'symptoms': 0.44788373658249625, 'likely_causes': 0.44860245116828357, 'diagnostic_steps': 1.8985348253693604, 'diy_or_mechanic': 1.4924207885021397}


    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=5
    )

    return results

In [27]:
evaluate(gt_test, lambda q: search(q["question"]))


  0%|          | 0/68 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.8713235294117647}

In [28]:
prompt2_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as 'NON_RELEVANT', 'PARTLY_RELEVANT', or 'RELEVANT'.

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [29]:
import json
from tqdm.auto import tqdm

df_sample = df_question.sample(n=10, random_state=1)
sample = df_sample.to_dict(orient="records")

evaluations = []

for record in tqdm(sample):
    question = record["question"]
    answer_llm = rag(question)

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt)
    evaluation = json.loads(evaluation)

    evaluations.append((record, answer_llm, evaluation))

  0%|          | 0/10 [00:00<?, ?it/s]

In [30]:
df_eval = pd.DataFrame(evaluations, columns=["record", "answer", "evaluation"])

df_eval["issue_id"] = df_eval.record.apply(lambda d: d["issue_id"])
df_eval["question"] = df_eval.record.apply(lambda d: d["question"])
df_eval["relevance"] = df_eval.evaluation.apply(lambda d: d["Relevance"])
df_eval["explanation"] = df_eval.evaluation.apply(lambda d: d["Explanation"])

df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.9
PARTLY_RELEVANT    0.1
Name: proportion, dtype: float64

In [31]:

import os
model_name = os.getenv("AI_MODEL")
out_path = f"data/rag-eval-groq/{model_name}.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_eval.to_csv(out_path, index=False)

